# Pandas Profiling / YData Profiling
### Automated Exploratory Data Analysis and Dataset Health Auditing

---

### Scope & Focus of This Notebook
- **Primary Topic**: Automated Exploratory Data Analysis using the `ydata-profiling` library (formerly `pandas-profiling`).
- **What You Will Learn**:
  1. Automated profiling: Instantiating and configuring a `ProfileReport` object.
  2. Standalone export: Saving full interactive reports to disk as self-contained HTML documents (`profile.to_file()`).
  3. Diagnostic warnings: Interpreting automated alerts regarding high cardinality, collinearity, constant columns, zeros, and missingness.
  4. Advanced associations: Overview of non-linear correlations (Spearman, Kendall, Phik $\phi_K$) generated by the profiling engine.
  5. Critical ML thinking: Why automated EDA accelerates initial discovery but never replaces human domain knowledge or manual validation.
- **What We Will NOT Cover Here**:
  - Step-by-step manual univariate, bivariate, and multivariate visualization techniques were taught in **`01_Univariate_Analysis`**, **`02_Bivariate_Analysis`**, and **`03_Multivariate_Analysis`**.
  - Feature engineering, scaling, and machine learning pipelines belong to **`Module 03: Feature Engineering`**.

---

## 1. Setup & Environment Verification

### What are we doing?
We import `pandas` and `ydata_profiling.ProfileReport` and load the Titanic dataset using robust relative path checking.

#### What do these libraries do here?
- `pandas`: Tabular data loading and DataFrame management.
- `ydata_profiling`: Automated profiling engine that generates HTML diagnostic dashboards summarizing distributions, correlations, and data quality warnings.


In [1]:
import pandas as pd
import os
from ydata_profiling import ProfileReport

# Robust path resolution
data_path = '../../data/titanic.csv' if os.path.exists('../../data/titanic.csv') else 'data/titanic.csv'
df = pd.read_csv(data_path)

print(f"Loaded Titanic dataset: {df.shape[0]} rows, {df.shape[1]} columns")


Loaded Titanic dataset: 891 rows, 12 columns


/var/folders/z8/bjhnq05d3ygdk3mnkcvm6ys80000gn/T/ipykernel_7551/1930097021.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## 2. Generating the Profile Report (`ProfileReport`)

### What are we doing?
We instantiate `ProfileReport` on the Titanic DataFrame with exploratory analysis enabled.

#### What does `ProfileReport` do?
`ProfileReport(df, title=..., explorative=True)` scans every column in the dataset, computing univariate distributions, quantiles, missingness matrices, pairwise correlations, and automatic anomaly warnings in a single call.

#### Why are we using it?
To obtain a comprehensive baseline health check of a new dataset within seconds rather than manually writing dozens of individual inspection commands.

#### Important Parameters:
- `df`: Input DataFrame to profile.
- `title` (str): Title displayed in the generated report header.
- `explorative` (bool): Enables deeper analysis including detailed quantile computations and advanced correlation matrices.

#### What does it return?
A `ProfileReport` object that can be displayed inline in Jupyter or serialized to disk as an HTML document.


In [2]:
# Initialize the automated profile report
profile = ProfileReport(
    df, 
    title="Titanic Dataset — Automated EDA Report", 
    explorative=True
)

print("ProfileReport object initialized successfully.")


ProfileReport object initialized successfully.


## 3. Exporting Standalone Interactive Reports (`to_file`)

### What are we doing?
We export the complete interactive report to disk as an HTML file.

#### What does `profile.to_file(output_path)` do?
Compiles all plots, tables, and warnings into a self-contained HTML document with embedded CSS and JavaScript that can be opened in any web browser without requiring a running Python kernel.

#### Why are we using it?
To create persistent, shareable audit documentation for project stakeholders, data teams, or portfolio records.

#### What is the implication?
The output file is written to `outputs/reports/titanic_profiling_report.html`, providing an interactive dashboard accessible offline.


In [3]:
# Destination directory setup
report_dir = '../../outputs/reports' if os.path.exists('../../outputs/reports') else ('outputs/reports' if os.path.exists('outputs/reports') else '.')
os.makedirs(report_dir, exist_ok=True)
report_file = os.path.join(report_dir, 'titanic_profiling_report.html')

# Export standalone HTML report
profile.to_file(report_file)

print(f"Report exported successfully: {report_file}")
print(f"File size: {os.path.getsize(report_file) / (1024*1024):.2f} MB")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|                                                   | 0/12 [00:00<?, ?it/s]

100%|████████████████████████████████████████| 12/12 [00:00<00:00, 2821.60it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Report exported successfully: ../../outputs/reports/titanic_profiling_report.html
File size: 4.57 MB


## 4. Interpreting Dataset Alerts & Warnings

Automated profiling engines categorize dataset characteristics into actionable alerts:

| Alert Type | Titanic Example | What It Means | Recommended ML Action |
| :--- | :--- | :--- | :--- |
| **High Cardinality** | `Name` (891 distinct), `Ticket` (681 distinct) | Almost every row contains a unique text string. | Drop identifier columns or engineer grouped features (e.g. extract title `"Mr"`, `"Mrs"`). |
| **High Missingness** | `Cabin` (77.1% missing) | Over 75% of observations have `NaN`. | Drop the feature or create a binary indicator feature (`HasCabin`). |
| **Zeros** | `Fare` (15 passengers paid £0.00) | A valid numerical feature contains true zeroes. | Verify whether zeroes are genuine promotional fares, crew, or missing data codes. |
| **High Correlation** | `Pclass` & `Fare` | Strong collinearity between predictors. | Be mindful of multicollinearity when training linear/logistic regression models. |


## 5. Critical Discussion: The Boundaries of Automated EDA

While automated profiling tools provide immense time savings, they cannot replace human domain intuition:

1. **No Domain Context**: The tool flags `PassengerId` as a clean integer feature with uniform distribution; human intuition recognizes it is a meaningless primary key that must be dropped.
2. **Cannot Formulate ML Targets**: The tool treats `Survived` as just another integer column; it does not know `Survived` is the prediction target.
3. **No Causal Reasoning**: The tool reports correlation matrices; it cannot deduce *why* 1st class females survived at 97% while 3rd class males survived at 13%.
4. **Data Leakage Ignorance**: The tool analyzes the dataset in aggregate; it cannot detect if information from the future (or test set) is leaking into training features.

> **Rule of Thumb**: Use automated profiling as your **Day 1 reconnaissance scout**, then switch to **manual targeted EDA** to build domain-specific features and validate hypotheses.


---

## What We Learned
- **Automated Profiling**: `ProfileReport` compiles comprehensive statistical summaries, quantiles, and nullity patterns in a single call.
- **Standalone Artifacts**: `profile.to_file()` exports interactive HTML reports for persistent project documentation.
- **Automated Alerts**: Profiling engines detect cardinality risks, zero-value concentrations, missingness, and collinearity.
- **The Role of Automation**: Automated tools accelerate initial data reconnaissance but cannot substitute for problem framing, target formulation, or causal validation.

---

## Important Takeaways
1. **Always export reports to HTML**: Sharing standalone HTML reports allows non-technical collaborators to explore data health without Python setup.
2. **Review the 'Alerts' tab first**: The alerts section highlights critical data quality defects (missingness, constants, severe cardinality) immediately.
3. **Automated EDA is a starting point, not the conclusion**: Use automated insights to decide where deep manual EDA is required.

---

## Common Mistakes
1. **Running automated profiling on massive datasets without subsampling**: Running `ProfileReport` on a 10-million-row dataset can hang your machine. Always profile a representative `df.sample(50000)` first.
2. **Blindly trusting automated advice**: An automated tool might recommend dropping a high-cardinality column like `Name`, missing the valuable title signals (`Dr`, `Master`, `Countess`) embedded inside it.
3. **Profiling after train/test split incorrectly**: Profiling the entire dataset together can subtly bias your modeling choices; profile the training split to avoid subtle mental data leakage.

---

## Final Mental Model

```text
┌────────────────────────────────────────────────────────────────────────┐
│                      DATA RECONNAISSANCE WORKFLOW                      │
└────────────────────────────────────────────────────────────────────────┘
          │
          ▼
   [ YData ProfileReport ] ──► Automated 360° health check
          │
          ▼
   [ Inspect Alerts Tab ]  ──► Identify: Missingness, Cardinality, Zeros, Collinearity
          │
          ▼
   [ Manual Targeted EDA ] ──► Verify domain logic, test feature interactions
          │
          ▼
   [ Model Pipeline ]     ──► Clean, encode, scale, and train
```
